# Note 01: First CUDA Program

CUDA C++ runs across a host CPU and a device GPU. This note starts with one minimal kernel, then builds the host and device mental model used throughout CUDA programming.

## 1. Check the GPU

First, make sure the Colab runtime is using a GPU:

**Runtime → Change runtime type → GPU**

Run:

```bash
!nvidia-smi
```

This shows the NVIDIA GPU attached to the current Colab runtime.

Then check the CUDA compiler:

```bash
!nvcc --version
```

`nvcc` is NVIDIA's CUDA compiler.

For normal C++ code, we commonly use:

```bash
g++
```

For CUDA C++, we use:

```bash
nvcc
```

CUDA source files normally use the `.cu` extension:

```text
hello.cu
```

instead of:

```text
hello.cpp
```

---

# 2. CUDA Programs Have Two Sides

A CUDA program usually contains code that runs in two different places:

```text
CPU = Host
GPU = Device
```

So one `.cu` file can contain:

```text
Host code
    +
Device code
```

The CPU controls the overall program.

The GPU performs highly parallel computation when the CPU asks it to.

A useful mental model is:

```text
CPU program
    |
    | launches work
    v
GPU
```

This is important:

CUDA does **not** mean that your entire program suddenly runs on the GPU.

Usually:

```text
CPU
- starts the program
- allocates memory
- prepares data
- launches kernels
- coordinates execution

GPU
- runs highly parallel kernels
- performs the heavy computation
```

---

# 3. Our First CUDA Program

Create a CUDA file:

```cpp
%%writefile hello.cu

#include <iostream>
#include <cuda_runtime.h>

__global__ void hello_gpu() {
    printf("Hello from GPU!\n");
}

int main() {

    std::cout << "Hello from CPU!\n";

    hello_gpu<<<1, 1>>>();

    cudaDeviceSynchronize();

    return 0;
}
```

Compile it:

```bash
!nvcc hello.cu -o hello
```

Run it:

```bash
!./hello
```

Expected output:

```text
Hello from CPU!
Hello from GPU!
```

---

# 4. Why Do We Use `std::cout` on the CPU but `printf()` on the GPU?

Notice that we wrote:

```cpp
std::cout << "Hello from CPU!\n";
```

inside the CPU code.

But inside the GPU kernel we wrote:

```cpp
printf("Hello from GPU!\n");
```

instead of:

```cpp
std::cout
```

This is intentional.

## Host code

The CPU side is normal C++.

So normal C++ facilities such as:

```cpp
std::cout
```

are completely fine.

Example:

```cpp
int main() {
    std::cout << "Running on CPU\n";
}
```

---

## Device code

Code inside a CUDA kernel runs on the GPU.

Standard C++ stream objects like:

```cpp
std::cout
```

are not normally available inside device code.

CUDA supports device-side:

```cpp
printf()
```

mainly for debugging.

So:

```cpp
__global__ void kernel() {
    printf("Hello from GPU\n");
}
```

works.

But this generally does not:

```cpp
__global__ void kernel() {
    std::cout << "Hello from GPU\n";
}
```

---

## Normal CUDA Style

A common pattern is therefore:

```text
Host / CPU code
    std::cout

Device / GPU code
    printf() only when debugging
```

But real performance kernels normally contain **no printing at all**.

For example:

```cpp
__global__ void add(
    const float* a,
    const float* b,
    float* c
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    c[i] = a[i] + b[i];
}
```

The kernel's purpose is computation, not printing.

Printing from thousands of GPU threads would also be extremely noisy and slow.

So GPU `printf()` should mostly be thought of as:

> a debugging tool

rather than something we normally use in production kernels.

---

# 5. Understanding `main()`

This:

```cpp
int main()
```

is a normal C++ function.

It runs on the **CPU**.

So this:

```cpp
std::cout << "Hello from CPU!\n";
```

is executed by the CPU.

The program begins here, just like a normal C++ program.

---

# 6. What Is `__global__`?

Now we reach the first CUDA-specific keyword:

```cpp
__global__ void hello_gpu()
```

`__global__` tells CUDA:

> This function runs on the GPU and can be launched from the CPU.

A function like this is called a:

# CUDA Kernel

So for now:

```text
Normal function
    ↓
usually runs on CPU


__global__ function
    ↓
runs on GPU
```

Example:

```cpp
__global__ void hello_gpu() {
    printf("Hello from GPU!\n");
}
```

The function itself is written in CUDA C++.

But when it executes, its instructions run on the GPU.

---

# 7. CPU Launches the GPU Kernel

The kernel is launched here:

```cpp
hello_gpu<<<1, 1>>>();
```

This looks strange because it is **CUDA-specific syntax**.

A normal C++ function call looks like:

```cpp
hello();
```

A CUDA kernel launch looks like:

```cpp
hello_gpu<<<..., ...>>>();
```

The:

```text
<<< >>>
```

syntax tells CUDA how much parallel GPU work we want to launch.

For now, do not worry about exactly what:

```cpp
<<<1, 1>>>
```

means.

We will understand it properly in the next part.

Conceptually, this line means:

```text
CPU
 |
 | launch hello_gpu
 v
GPU
 |
 | execute kernel
 v
finish
```

---

# 8. Why Are the CPU and GPU Functions Written Separately?

This is one of the most important CUDA ideas.

We cannot simply write:

```cpp
int main() {
    // somehow everything runs on GPU
}
```

because the CPU and GPU are separate processors.

They have different roles.

So we explicitly separate:

```cpp
int main()
```

which runs on the CPU,

from:

```cpp
__global__ void hello_gpu()
```

which runs on the GPU.

The CPU controls the GPU.

Conceptually:

```text
Host / CPU
    |
    | launch kernel
    v
Device / GPU
```

This pattern will appear constantly.

Later, an inference program might look like:

```text
CPU inference runtime
        |
        | launches
        v

GPU kernel: RMSNorm
        |
GPU kernel: GEMM
        |
GPU kernel: Attention
        |
GPU kernel: Sampling
```

The CPU runtime decides **what needs to happen**.

The GPU kernels perform the expensive parallel computation.

---

# 9. CUDA Kernel Launches Are Asynchronous

Now consider:

```cpp
hello_gpu<<<1, 1>>>();
```

You might assume the CPU waits here until the GPU finishes.

Usually, it does not.

CUDA kernel launches are normally **asynchronous** with respect to the CPU.

Conceptually:

```text
CPU:
"GPU, run this kernel."

GPU:
"Okay."

CPU:
"I'm continuing with the rest of my program."
```

The CPU can continue executing while the GPU is working.

This is extremely important for CUDA performance.

It lets us eventually overlap work instead of constantly making the CPU and GPU wait for each other.

---

# 10. `cudaDeviceSynchronize()`

This line:

```cpp
cudaDeviceSynchronize();
```

changes that behavior.

It tells the CPU:

> Wait here until the GPU has finished all previously submitted work.

So:

```cpp
hello_gpu<<<1, 1>>>();

cudaDeviceSynchronize();
```

means:

```text
CPU launches GPU kernel
        ↓
GPU begins working
        ↓
CPU reaches synchronization
        ↓
CPU waits
        ↓
GPU finishes
        ↓
CPU continues
```

For our simple hello-world program, this ensures the GPU has finished printing before the program exits.

Later, unnecessary synchronization can hurt performance, so we won't randomly throw:

```cpp
cudaDeviceSynchronize();
```

everywhere.

For now, we use it because we explicitly want to wait for the kernel.

---

# 11. What Does `#include <cuda_runtime.h>` Do?

We included:

```cpp
#include <cuda_runtime.h>
```

This gives us access to CUDA Runtime API functions such as:

```cpp
cudaDeviceSynchronize()
```

Later we'll use many more functions such as:

```cpp
cudaMalloc()
cudaFree()
cudaMemcpy()
cudaGetLastError()
cudaEventCreate()
```

These are functions provided by the CUDA Runtime.

---

# 12. What Does `nvcc` Do?

We compiled the program using:

```bash
nvcc hello.cu -o hello
```

`nvcc` is the NVIDIA CUDA compiler driver.

Our `.cu` file contains both:

```text
CPU code
+
GPU code
```

So CUDA's compilation process has to handle both sides.

At a high level:

```text
hello.cu
   |
   v
nvcc
   |
   +------ CPU code
   |
   +------ GPU code
   |
   v
executable
```

We will later learn the actual CUDA compilation pipeline:

```text
CUDA C++
   ↓
PTX
   ↓
GPU machine code
```

But we do not need that yet.

---

# 13. Current Mental Model

At this point, this is the main picture to remember:

```text
             HOST
             CPU
              |
              |
      int main()
              |
              |
      kernel launch
   hello_gpu<<<...>>>()
              |
              v
          DEVICE
           GPU
              |
              |
   __global__ hello_gpu()
              |
              v
        GPU executes work
```

CPU = controller.

GPU = parallel worker.

---

# 14. What We Learned

We now know:

* CUDA source files normally use `.cu`
* `nvcc` compiles CUDA C++
* CPU is called the **host**
* GPU is called the **device**
* a CUDA program usually contains both host and device code
* `main()` runs on the CPU
* `std::cout` is normal for host-side C++
* device-side `printf()` is useful mainly for debugging
* production GPU kernels normally do not print anything
* `__global__` defines a GPU kernel callable from the host
* `<<< >>>` launches a CUDA kernel
* the CPU controls the overall CUDA program
* the GPU executes kernels
* kernel launches are usually asynchronous
* `cudaDeviceSynchronize()` makes the CPU wait for the GPU
* `cuda_runtime.h` exposes the CUDA Runtime API

---

# Next — CUDA Threads, Blocks and Grids

The next thing we need to understand is what this actually means:

```cpp
hello_gpu<<<1, 1>>>();
```

Why are there two numbers?

Why does CUDA need them?

And how do we go from:

```text
1 GPU thread
```

to:

```text
thousands or millions of GPU threads
```

That introduces the fundamental CUDA execution hierarchy:

```text
Grid
 ↓
Blocks
 ↓
Threads
```


In [ ]:
!nvidia-smi

Sat Oct 31 14:10:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvcc --version #cuda complier , c ++ has g++

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [ ]:
%%writefile hello.cu
#include <iostream>
#include <cuda_runtime.h>

__global__ void hello_gpu() {
    printf("Hello from GPU!\n");
}

int main() {
    std::cout << "Hello from CPU!\n";

    hello_gpu<<<1, 1>>>();

    cudaDeviceSynchronize();

    return 0;
}

Writing hello.cu


In [ ]:
!nvcc hello.cu -o hello
!./hello

Hello from CPU!
Hello from GPU!
